<a href="https://colab.research.google.com/github/ismethakanaydogmus/News_Classification/blob/main/News_Classification_V3_DeepLearning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🧠 News Classification V3.0: Deep Learning (BERTurk)

Bu aşamada, klasik makine öğrenmesi sınırlarını aşarak **Transformers** mimarisine geçiş yapıyoruz. Hedefimiz, kelimelerin köklerinden ziyade cümle içindeki anlamsal bağlamlarını (context) öğrenmek.

In this stage, we are moving beyond classical machine learning to the **Transformers** architecture. Our goal is to learn the semantic context of words rather than just their roots.

In [6]:
# 1. Projeyi klonla ve dizine gir
!git clone https://github.com/ismethakanaydogmus/News_Classification.git
%cd News_Classification

# 2. V3 için gerekli "Hugging Face" kütüphanelerini kur
!pip install transformers[torch] datasets accelerate -U

import pandas as pd
import torch
from datasets import Dataset

# 3. GPU Kontrolü (V3.0 için hayati önem taşır!)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🚀 Model şu cihazda çalışacak: {device}")
if device == "cuda":
    print(f"GPU Model: {torch.cuda.get_device_name(0)}")

Cloning into 'News_Classification'...
remote: Enumerating objects: 38, done.
remote: Counting objects: 100% (38/38), done.
remote: Compressing objects: 100% (31/31), done.
remote: Total 38 (delta 14), reused 24 (delta 6), pack-reused 0 (from 0)
Receiving objects: 100% (38/38), 580.95 KiB | 2.74 MiB/s, done.
Resolving deltas: 100% (14/14), done.
/content/News_Classification/News_Classification
🚀 Model şu cihazda çalışacak: cuda
GPU Model: Tesla T4


## 🛠️ 1. Veri Hazırlama ve Etiketleme / Data Preparation

BERT modeli kategorileri (Ekonomi, Siyaset vb.) metin olarak değil, sayısal ID'ler olarak bekler. Bu aşamada:
1. Kategorileri benzersiz ID'lere eşleyeceğiz (`label2id`).
2. Veriyi Pandas formatından Hugging Face'in optimize edilmiş `Dataset` formatına çevireceğiz.

---

BERT expects categories as numerical IDs. We will map categories to unique IDs and convert the Pandas DataFrame into the highly optimized Hugging Face `Dataset` format.

In [7]:
import pandas as pd
from datasets import Dataset

# 1. Veriyi yükle
df = pd.read_csv("master_news_dataset_v2.csv")

# 2. Kategori -> ID eşleşmesi (Label Mapping)
categories = sorted(df['category'].unique())
label2id = {label: i for i, label in enumerate(categories)}
id2label = {i: label for i, label in enumerate(categories)}

print(f"✅ Kategoriler ve ID karşılıkları:\n{label2id}")

# DataFrame'e sayısal etiketleri ekleyelim
df['label'] = df['category'].map(label2id)

# 3. Hugging Face Dataset formatına çevirme
# Sadece gerekli olan başlık ve etiket sütunlarını alıyoruz
raw_dataset = Dataset.from_pandas(df[['headline', 'label']])

# Eğitim (%80) ve Test (%20) olarak bölelim
dataset_dict = raw_dataset.train_test_split(test_size=0.2, seed=42)

print("\n✅ Veri seti bölündü:")
print(dataset_dict)

✅ Kategoriler ve ID karşılıkları:
{'Astroloji': 0, 'Bilim': 1, 'Dunya': 2, 'Egitim': 3, 'Ekonomi': 4, 'Kultur-Sanat': 5, 'Saglik': 6, 'Siyaset': 7, 'Spor': 8, 'Teknoloji': 9}

✅ Veri seti bölündü:
DatasetDict({
    train: Dataset({
        features: ['headline', 'label'],
        num_rows: 4572
    })
    test: Dataset({
        features: ['headline', 'label'],
        num_rows: 1144
    })
})


## 🔠 2. Tokenization: BERTurk ile Metin İşleme

BERT, kelimeleri doğrudan okumak yerine onları **Token** adı verilen parçalara böler. Bizim kullanacağımız **dbmdz/bert-base-turkish-cased** modeli, Türkçe'nin eklemeli yapısına (agglutinative) çok uygun olan **WordPiece** yöntemini kullanır.

* **Neden Önemli?** Bu yöntem sayesinde model "Gidemedim" kelimesini `["git", "##emedim"]` gibi parçalayarak kelimenin kökündeki eylemi ve ekin kattığı olumsuzluk/zaman anlamını ayrı ayrı kavrar.

---

BERT breaks down text into **Tokens** using the **WordPiece** method, which is ideal for Turkish. This allows the model to understand the semantic meaning of suffixes and complex word structures.

In [8]:
from transformers import AutoTokenizer

# Türkçe BERT modelinin tokenizer'ını yükleyelim
model_checkpoint = "dbmdz/bert-base-turkish-cased"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

def tokenize_function(examples):
    # padding: Cümleleri 64 karakter boyuna tamamlar
    # truncation: 64 karakterden uzun olanları keser
    return tokenizer(examples["headline"], padding="max_length", truncation=True, max_length=64)

# Tüm veri setini tokenize edelim (Map fonksiyonu çok hızlıdır)
tokenized_datasets = dataset_dict.map(tokenize_function, batched=True)

# Orijinal metin sütununa artık ihtiyacımız yok, sayısal versiyonlar yeterli
tokenized_datasets = tokenized_datasets.remove_columns(["headline"])

print("✅ Tokenization tamamlandı! Model için hazırlanan örnek veri yapısı:")
print(tokenized_datasets["train"][0].keys())

Map:   0%|          | 0/4572 [00:00<?, ? examples/s]

Map:   0%|          | 0/1144 [00:00<?, ? examples/s]

✅ Tokenization tamamlandı! Model için hazırlanan örnek veri yapısı:
dict_keys(['label', 'input_ids', 'token_type_ids', 'attention_mask'])


## 🧠 3. Model Hazırlığı ve İnce Ayar / Model & Fine-Tuning

Şimdi, milyonlarca Türkçe cümle ile önceden eğitilmiş olan **BERTurk** modelini çağıracağız. Ancak bir farkla: Modelin en üst katmanını (head) söküp, yerine bizim 10 kategorimizi (Ekonomi, Astroloji vb.) sınıflandıracak yeni bir katman takacağız.

### ⚙️ Hiper-parametreler / Hyper-parameters
* **Learning Rate (2e-5):** Derin öğrenmede "öğrenme hızı" çok kritiktir. Çok yüksek olursa model şaşırır, çok düşük olursa yerinde sayar. 2e-5, BERT için altın orandır.
* **Epoch (3):** Tüm veri setini modelin önüne 3 kez getireceğiz. Fazlası "ezberleme" (overfitting) riskine yol açabilir.
* **Batch Size (16):** Verileri 16'şarlı gruplar halinde modele vereceğiz.

---

We will initialize the pre-trained **BERTurk** model and replace its classification head to match our 10 categories. We will use a learning rate of **2e-5** and train for **3 epochs**, which is the standard recipe for stable fine-tuning.

In [9]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

# 1. Modeli yükleyelim
model = AutoModelForSequenceClassification.from_pretrained(
    model_checkpoint,
    num_labels=len(categories), # 10 kategori
    id2label=id2label,
    label2id=label2id
).to(device)

# 2. Başarı metriklerini tanımlayalım
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, predictions)
    f1 = f1_score(labels, predictions, average="weighted")
    return {"accuracy": acc, "f1": f1}

print(f"✅ {model_checkpoint} modeli {len(categories)} sınıf için hazırlandı!")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: dbmdz/bert-base-turkish-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


✅ dbmdz/bert-base-turkish-cased modeli 10 sınıf için hazırlandı!


In [12]:
from transformers import Trainer

# Trainer'ı oluştur - Transformers v5+ Uyumlu
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    processing_class=tokenizer,       # 'tokenizer' yerine 'processing_class' yazıyoruz
    compute_metrics=compute_metrics,
)

# 🏁 EĞİTİMİ BAŞLAT / START TRAINING
print("🔥 V3.0 Fine-tuning (BERT) başlatıldı. T4 GPU devrede...")
trainer.train()

🔥 V3.0 Fine-tuning (BERT) başlatıldı. T4 GPU devrede...


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.747806,0.619201,0.813811,0.813627
2,0.510486,0.577788,0.819930,0.819696
3,0.387583,0.583214,0.822552,0.822334


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

TrainOutput(global_step=858, training_loss=0.7152536715660895, metrics={'train_runtime': 222.5882, 'train_samples_per_second': 61.621, 'train_steps_per_second': 3.855, 'total_flos': 451136306654208.0, 'train_loss': 0.7152536715660895, 'epoch': 3.0})

## 📊 4. V3.0 Performans Değerlendirmesi / Performance Evaluation

V3.0 (BERTurk) modelimiz, klasik yöntemlerin (LinearSVC) tıkandığı noktada bayrağı devraldı ve başarıyı **%82** seviyesine taşıdı.

### 🌟 Ana Gözlemler / Key Observations
* **İstikrarlı Öğrenme:** Training Loss her epoch'ta (0.74 -> 0.51 -> 0.38) düzenli olarak düştü. Bu, modelin ezberlemediğini, gerçekten öğrendiğini gösteriyor.
* **Bağlamsal Zeka:** Zeyrek gibi manuel kök bulma araçları kullanmamıza gerek kalmadan model, kelime eklerinden anlam çıkarmayı başardı.
* **Genelleme Yeteneği:** Validation Accuracy (**%82.3**), modelin hiç görmediği haber başlıklarını bile çok yüksek bir isabetle bildiğini kanıtlıyor.

---

V3.0 (BERTurk) outperformed classical methods by pushing accuracy to **82.3%**. The steady decrease in Training Loss (0.74 to 0.38) indicates healthy learning. The model successfully leveraged contextual understanding without the need for manual lemmatization.

In [13]:
from sklearn.metrics import classification_report

# Test seti üzerinde tahminleri alalım
predictions = trainer.predict(tokenized_datasets["test"])
preds = np.argmax(predictions.predictions, axis=-1)

print("\n--- V3.0 BERTurk Model Detaylı Performans Raporu ---")
print(classification_report(tokenized_datasets["test"]["label"], preds, target_names=categories))


--- V3.0 BERTurk Model Detaylı Performans Raporu ---
              precision    recall  f1-score   support

   Astroloji       0.96      0.99      0.98       109
       Bilim       0.78      0.70      0.74        98
       Dunya       0.67      0.68      0.68       109
      Egitim       0.95      0.91      0.93        58
     Ekonomi       0.75      0.81      0.78       148
Kultur-Sanat       0.69      0.75      0.72        79
      Saglik       0.82      0.84      0.83       157
     Siyaset       0.83      0.79      0.81       133
        Spor       0.95      0.96      0.96       137
   Teknoloji       0.80      0.74      0.77       116

    accuracy                           0.82      1144
   macro avg       0.82      0.82      0.82      1144
weighted avg       0.82      0.82      0.82      1144



In [18]:
def predict_v3(text):
    # Metni tokenize et
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=64).to(device)

    # Tahmin yap
    with torch.no_grad():
        outputs = model(**inputs)

    # En yüksek skora sahip etiketi bul
    probs = torch.nn.functional.softmax(outputs.logits, dim=-1)
    pred_idx = torch.argmax(probs).item()

    return id2label[pred_idx]

# Canlı Test
test_headlines = [
    "Merkür retrosu bu hafta tüm burçları derinden etkileyecek",
    "Şampiyonlar Ligi finalinde dev randevu: Real Madrid vs City",
    "Yapay zeka modelleri yazılım dünyasında yeni bir devir açıyor",
    "Yeni bulunan antik kent arkeoloji dünyasını heyecanlandırdı",
    "Okullarda yeni müfredat çalışmaları tamamlanmak üzere",
    "Mars'ta su izine rastlanması bilim insanlarını şaşırttı",
    "Keskin kararlar zamanı",
    "Motorine bu gece indirim geliyor! İşte yeni fiyatlar...",
]

print(f"{'HABER BAŞLIĞI':<60} | {'V3.0 TAHMİN'}")
print("-" * 80)
for h in test_headlines:
    print(f"{h[:58]:<60} | {predict_v3(h)}")

HABER BAŞLIĞI                                                | V3.0 TAHMİN
--------------------------------------------------------------------------------
Merkür retrosu bu hafta tüm burçları derinden etkileyecek    | Astroloji
Şampiyonlar Ligi finalinde dev randevu: Real Madrid vs Cit   | Spor
Yapay zeka modelleri yazılım dünyasında yeni bir devir açı   | Teknoloji
Yeni bulunan antik kent arkeoloji dünyasını heyecanlandırd   | Bilim
Okullarda yeni müfredat çalışmaları tamamlanmak üzere        | Egitim
Mars'ta su izine rastlanması bilim insanlarını şaşırttı      | Bilim
Keskin kararlar zamanı                                       | Astroloji
Motorine bu gece indirim geliyor! İşte yeni fiyatlar...      | Ekonomi
